# 03 — NLP Pipeline

Walk through the full NLP pipeline:
1. LDA topic modeling — what kinds of resistance exist?
2. VADER sentiment with domain lexicon
3. Vocal minority correction — population-adjusted scores
4. Before/after comparison for the methodology page

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Load posts with topic weights
posts_path = '../../data/processed/posts_with_topics.parquet'
posts = pd.read_parquet(posts_path)
print(f'Posts: {len(posts)}')
print(f'Columns: {list(posts.columns)}')

In [ ]:
# Topic distribution
if 'dominant_topic' in posts.columns:
    topic_summary = pd.read_csv('../../outputs/topic_summary.csv')
    print('=== LDA Topic Summary ===')
    for _, row in topic_summary.iterrows():
        print(f"\nTopic {row['topic_id']}: {row['label']}")
        print(f"  Top words: {row['top_words'][:80]}...")
    
    fig, ax = plt.subplots(figsize=(10, 5))
    counts = posts['dominant_topic'].value_counts().sort_index()
    labels = [topic_summary.iloc[i]['label'] if i < len(topic_summary) else f'Topic {i}'
              for i in counts.index]
    counts.plot.bar(ax=ax, color='#22c55e')
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_title('Post Distribution Across Topics')
    ax.set_ylabel('Number of posts')
    plt.tight_layout()
    plt.show()

In [ ]:
# Sentiment distribution
from analytics.nlp.sentiment import compute_post_sentiment

if 'sentiment' not in posts.columns:
    posts = compute_post_sentiment(posts)

fig, ax = plt.subplots(figsize=(10, 4))
posts['sentiment'].hist(bins=50, ax=ax, color='#3b82f6', alpha=0.7)
ax.axvline(0, color='white', linestyle='--', alpha=0.5)
ax.set_title('VADER Sentiment Distribution (with domain lexicon)')
ax.set_xlabel('Compound sentiment (-1 = negative, +1 = positive)')
plt.tight_layout()
plt.show()

print(f'Mean sentiment: {posts["sentiment"].mean():.3f}')
print(f'Negative posts (<-0.05): {(posts["sentiment"] < -0.05).sum()} ({(posts["sentiment"] < -0.05).mean()*100:.1f}%)')
print(f'Positive posts (>+0.05): {(posts["sentiment"] > 0.05).sum()} ({(posts["sentiment"] > 0.05).mean()*100:.1f}%)')

In [ ]:
# Vocal minority correction — before vs after
resistance_path = '../../data/processed/county_resistance_scores.parquet'
resistance = pd.read_parquet(resistance_path)

fips_col = 'matched_county_fips' if 'matched_county_fips' in resistance.columns else 'county_fips'

print('=== VOCAL MINORITY CORRECTION ===')
print('\nTop 5 by RAW resistance (before correction):')
print(resistance.nlargest(5, 'raw_resistance_mean')[
    [fips_col, 'raw_resistance_mean', 'post_count', 'population']
].to_string(index=False))

print('\nTop 5 by CORRECTED resistance (after vocal minority adjustment):')
print(resistance.nlargest(5, 'social_resistance_score')[
    [fips_col, 'social_resistance_score', 'post_count', 'population']
].to_string(index=False))